# Model B — Distractor & Hint Generator
**Course:** Artificial Intelligence — BS(CS) Spring 2026  
**Institution:** NUCES FAST, Islamabad  
**Notebook:** 3 of 5 — Model B Training

---
### What this notebook covers
| Sub-task | Approach |
|---|---|
| Distractor Candidate Extraction | String matching + frequency-based selection |
| Distractor Ranking | OHE + Cosine Similarity + Logistic Regression / RF |
| Diversity Penalty | Ensures 3 distinct distractors |
| Word2Vec Nearest Neighbours | Pre-trained Google-News vectors |
| Hint Extraction (Extractive) | Cosine-scored sentence ranking |
| Hint Extraction (ML-scored) | Logistic Regression on sentence features |
| Evaluation | Precision, Recall, F1, Confusion Matrix, R² Score |

**Prerequisite:** `1_preprocessing.ipynb` must have been run.

## 0. Imports & Config

In [1]:
import os, re, warnings, joblib, time
import numpy as np
import pandas as pd
import scipy.sparse as sp
from collections import Counter

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    confusion_matrix, r2_score, mean_squared_error
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
np.random.seed(42)

PROC = '/kaggle/input/datasets/muhammadibrahim304/race-processed-dataset'
MOUT = '/kaggle/working/models/model_b'
os.makedirs(MOUT, exist_ok=True)
print('Model B output dir:', MOUT)

Model B output dir: /kaggle/working/models/model_b


## 1. Load Data

In [2]:
train_df = pd.read_csv(f'{PROC}/train_processed.csv')
val_df   = pd.read_csv(f'{PROC}/val_processed.csv')
test_df  = pd.read_csv(f'{PROC}/test_processed.csv')
ohe_vec  = joblib.load(f'{PROC}/ohe_vectorizer.pkl')
le       = joblib.load(f'{PROC}/label_encoder.pkl')

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

STOP_WORDS = set("a an the is was are were be been being have has had do does did "
                 "will would could should may might shall can must of in on at to "
                 "for with by from up about into through during before after i me "
                 "my we our you your he she it they them their its this that".split())

Train: 61,484  Val: 8,784  Test: 17,571


## 2. Distractor Candidate Extraction

**Step 1:** Retrieve candidate phrases from the passage using frequency-based  
word selection (classical ML — no external NLP tools).

In [3]:
def extract_candidate_phrases(article_clean: str, answer_clean: str,
                               window: int = 3, top_n: int = 20) -> list:
    """
    Extract candidate distractor phrases from an article.
    Strategy: sliding window of `window` words; keep phrases not in answer;
    rank by frequency and content-word overlap difference.
    """
    words = article_clean.split()
    answer_tokens = set(answer_clean.split())
    candidates = []

    # Unigram & bigram phrases
    for n in range(1, window + 1):
        for i in range(len(words) - n + 1):
            phrase = ' '.join(words[i:i + n])
            phrase_tokens = set(phrase.split())
            # Skip if it's the answer or a stop-word-only phrase
            content_tokens = phrase_tokens - STOP_WORDS
            if not content_tokens:
                continue
            overlap_with_ans = len(phrase_tokens & answer_tokens) / (len(phrase_tokens) + 1)
            if overlap_with_ans < 0.5:  # not too similar to correct answer
                candidates.append(phrase)

    # Rank by frequency
    freq = Counter(candidates)
    top = [phrase for phrase, _ in freq.most_common(top_n)]
    return top

# Demo
row = train_df.iloc[0]
correct_opt = row['answer']
answer_clean = row[f'{correct_opt}_clean']
cands = extract_candidate_phrases(row['article_clean'], answer_clean)
print(f"Extracted {len(cands)} candidate phrases:")
for c in cands[:8]:
    print(f"  '{c}'")

Extracted 20 candidate phrases:
  'fish'
  'people'
  'watched'
  'picture'
  'way'
  'and'
  'other'
  'humans'


## 3. Distractor Feature Engineering

**Step 2:** For each candidate compute:
- OHE cosine similarity to correct answer
- Character-level match score
- Passage frequency
- Length features

In [4]:
def candidate_features(candidate: str, answer_clean: str,
                        article_clean: str, vec) -> np.ndarray:
    """Feature vector for a single distractor candidate."""
    # OHE cosine similarity to answer
    v_cand   = vec.transform([candidate])
    v_ans    = vec.transform([answer_clean])
    v_art    = vec.transform([article_clean])

    cos_ans  = float(cosine_similarity(v_cand, v_ans)[0, 0])
    cos_art  = float(cosine_similarity(v_cand, v_art)[0, 0])

    # Character-level match (longest common substring ratio)
    s1, s2 = candidate, answer_clean
    shorter = min(len(s1), len(s2)) + 1
    char_match = sum(a == b for a, b in zip(s1, s2)) / shorter

    # Passage frequency (rough)
    freq = article_clean.count(candidate) / (len(article_clean.split()) + 1)

    # Length features
    len_words = len(candidate.split())
    len_chars = len(candidate)
    ans_len_words = len(answer_clean.split())
    len_ratio = len_words / (ans_len_words + 1)

    # Starts with same token as answer
    same_start = int(candidate[:3] == answer_clean[:3])

    return np.array([cos_ans, cos_art, char_match, freq,
                     len_words, len_chars, len_ratio, same_start])

FEAT_COLS = ['cos_ans', 'cos_art', 'char_match', 'freq',
             'len_words', 'len_chars', 'len_ratio', 'same_start']

print(f"Feature dimension: {len(FEAT_COLS)}")
# Demo feature
f = candidate_features(cands[0], answer_clean, row['article_clean'], ohe_vec)
print(dict(zip(FEAT_COLS, f.round(4))))

Feature dimension: 8
{'cos_ans': np.float64(0.0), 'cos_art': np.float64(0.0836), 'char_match': np.float64(0.0), 'freq': np.float64(0.0365), 'len_words': np.float64(1.0), 'len_chars': np.float64(4.0), 'len_ratio': np.float64(0.1429), 'same_start': np.float64(0.0)}


## 4. Build Distractor Ranker Training Dataset

**Label**: 1 if the candidate IS one of the actual wrong options (A/B/C/D minus correct);  
0 otherwise. We then train a classifier to score each candidate.

In [6]:
print("Building distractor ranker dataset (sample 5000 rows) ...")
RANKER_SAMPLE = 5000

X_ranker, y_ranker, meta_rows = [], [], []

for _, row in train_df.head(RANKER_SAMPLE).iterrows():
    correct_opt   = row['answer']
    
    # Ensure answer_clean is a string to prevent similar issues
    answer_clean  = str(row[f'{correct_opt}_clean']) if pd.notna(row[f'{correct_opt}_clean']) else ""
    article_clean = str(row['article_clean']) if pd.notna(row['article_clean']) else ""

    # The 3 actual wrong options are the gold distractors
    gold_distractors = set()
    for opt in ['A', 'B', 'C', 'D']:
        if opt != correct_opt:
            val = row[f'{opt}_clean']
            # FIX: Only add to gold distractors if it is a valid string
            if isinstance(val, str):
                gold_distractors.add(val)

    candidates = extract_candidate_phrases(article_clean, answer_clean, top_n=15)

    for cand in candidates:
        # Also ensure candidate is a string just to be safe
        cand = str(cand)
        feats = candidate_features(cand, answer_clean, article_clean, ohe_vec)
        
        # Label: 1 if this candidate closely matches a gold distractor
        label = 0
        for gd in gold_distractors:
            cand_words = set(cand.split())
            gd_words   = set(gd.split())
            if len(cand_words & gd_words) / (len(cand_words | gd_words) + 1e-9) > 0.5:
                label = 1
                break
                
        X_ranker.append(feats)
        y_ranker.append(label)
        meta_rows.append({'candidate': cand, 'answer': answer_clean, 'label': label})

X_ranker = np.array(X_ranker)
y_ranker = np.array(y_ranker)
print(f"Distractor ranker dataset: {X_ranker.shape}, "
      f"positive rate: {y_ranker.mean():.3f}")

Building distractor ranker dataset (sample 5000 rows) ...
Distractor ranker dataset: (75000, 8), positive rate: 0.002


## 5. Train Distractor Ranker — Logistic Regression & Random Forest

In [7]:
from sklearn.model_selection import train_test_split

Xr_tr, Xr_val, yr_tr, yr_val = train_test_split(
    X_ranker, y_ranker, test_size=0.2, random_state=42, stratify=y_ranker
)

# Logistic Regression ranker
print("Training LR distractor ranker ...")
lr_ranker = LogisticRegression(C=1.0, max_iter=500, class_weight='balanced',
                                random_state=42)
lr_ranker.fit(Xr_tr, yr_tr)
yr_pred_lr = lr_ranker.predict(Xr_val)

lr_prec = precision_score(yr_val, yr_pred_lr, zero_division=0)
lr_rec  = recall_score(yr_val, yr_pred_lr, zero_division=0)
lr_f1   = f1_score(yr_val, yr_pred_lr, zero_division=0)
lr_acc  = accuracy_score(yr_val, yr_pred_lr)
print(f"  LR Ranker — Prec={lr_prec:.4f}  Rec={lr_rec:.4f}  "
      f"F1={lr_f1:.4f}  Acc={lr_acc:.4f}")

# Random Forest ranker
print("Training RF distractor ranker ...")
rf_ranker = RandomForestClassifier(n_estimators=100, max_depth=8,
                                    class_weight='balanced', n_jobs=-1,
                                    random_state=42)
rf_ranker.fit(Xr_tr, yr_tr)
yr_pred_rf = rf_ranker.predict(Xr_val)

rf_prec = precision_score(yr_val, yr_pred_rf, zero_division=0)
rf_rec  = recall_score(yr_val, yr_pred_rf, zero_division=0)
rf_f1   = f1_score(yr_val, yr_pred_rf, zero_division=0)
rf_acc  = accuracy_score(yr_val, yr_pred_rf)
print(f"  RF Ranker — Prec={rf_prec:.4f}  Rec={rf_rec:.4f}  "
      f"F1={rf_f1:.4f}  Acc={rf_acc:.4f}")

joblib.dump(lr_ranker, f'{MOUT}/distractor_lr_ranker.pkl')
joblib.dump(rf_ranker, f'{MOUT}/distractor_rf_ranker.pkl')
print("Rankers saved.")

Training LR distractor ranker ...
  LR Ranker — Prec=0.0097  Rec=0.7812  F1=0.0191  Acc=0.8287
Training RF distractor ranker ...
  RF Ranker — Prec=0.0112  Rec=0.5000  F1=0.0219  Acc=0.9048
Rankers saved.


## 6. Full Distractor Generation Pipeline

Combining Steps 1–3: extract → rank → diversity penalty → select top-3.

In [8]:
def diversity_penalty(candidate: str, selected: list) -> float:
    """Return max token overlap with already-selected distractors."""
    if not selected:
        return 0.0
    c_words = set(candidate.split())
    max_overlap = 0.0
    for sel in selected:
        s_words = set(sel.split())
        overlap = len(c_words & s_words) / (len(c_words | s_words) + 1e-9)
        max_overlap = max(max_overlap, overlap)
    return max_overlap

def generate_distractors(article_clean: str, answer_clean: str,
                          ranker, vec, n_distractors: int = 3,
                          diversity_threshold: float = 0.5) -> list:
    """
    Full pipeline: extract candidates → feature engineering →
    rank with ML model → apply diversity penalty → return top-N.
    """
    candidates = extract_candidate_phrases(article_clean, answer_clean, top_n=30)
    if not candidates:
        return []

    feats = np.array([
        candidate_features(c, answer_clean, article_clean, vec)
        for c in candidates
    ])

    # Score: probability of being a good distractor
    if hasattr(ranker, 'predict_proba'):
        scores = ranker.predict_proba(feats)[:, 1]
    else:
        scores = ranker.predict(feats).astype(float)

    # Sort by score descending
    order = np.argsort(scores)[::-1]
    selected = []
    for idx in order:
        cand = candidates[idx]
        if cand == answer_clean:
            continue
        if diversity_penalty(cand, selected) < diversity_threshold:
            selected.append(cand)
        if len(selected) == n_distractors:
            break

    # Pad if insufficient candidates found
    while len(selected) < n_distractors:
        selected.append('[distractor unavailable]')
    return selected

# Demo
row = val_df.iloc[5]
correct_opt  = row['answer']
answer_clean = row[f'{correct_opt}_clean']

distractors = generate_distractors(
    row['article_clean'], answer_clean, rf_ranker, ohe_vec
)
print("=== Distractor Generation Demo ===")
print(f"Correct answer : {answer_clean[:80]}")
print(f"Gold distractors:")
for opt in ['A','B','C','D']:
    if opt != correct_opt:
        print(f"  [{opt}] {row[f'{opt}_clean'][:80]}")
print("Generated distractors:")
for i, d in enumerate(distractors):
    print(f"  [D{i+1}] {d[:80]}")

=== Distractor Generation Demo ===
Correct answer : made up
Gold distractors:
  [A] made into
  [B] made for
  [D] were made up of
Generated distractors:
  [D1] and cities
  [D2] the land
  [D3] towns and


## 7. (Optional) Word2Vec Nearest-Neighbour Distractors

Uses pre-trained Gensim Word2Vec if the `word2vec-google-news-300` model  
is available in the Kaggle input. Skip gracefully if not present.

In [9]:
W2V_PATH = '/kaggle/input/googlenewsvectors/GoogleNews-vectors-negative300.bin'

w2v_model = None
if os.path.exists(W2V_PATH):
    from gensim.models import KeyedVectors
    print("Loading Word2Vec model (this takes ~3 min) ...")
    w2v_model = KeyedVectors.load_word2vec_format(W2V_PATH, binary=True)
    print(f"Word2Vec vocab size: {len(w2v_model.key_to_index):,}")
else:
    print("Word2Vec model not found at expected path. Skipping W2V distractors.")
    print("To enable: add 'GoogleNews vectors negative300' dataset to Kaggle input.")

def w2v_distractors(answer: str, article_clean: str,
                     model, top_n: int = 20, n_keep: int = 3) -> list:
    """Return top-N nearest neighbours not extractable from the passage."""
    if model is None:
        return []
    # Use first content token as pivot
    tokens = [t for t in answer.split() if t in model.key_to_index and t not in STOP_WORDS]
    if not tokens:
        return []
    pivot = tokens[0]
    neighbours = [word for word, _ in model.most_similar(pivot, topn=top_n)]
    # Filter out words appearing in the passage
    article_words = set(article_clean.split())
    out_of_passage = [n for n in neighbours if n.lower() not in article_words]
    return out_of_passage[:n_keep]

if w2v_model:
    row = val_df.iloc[5]
    correct_opt = row['answer']
    answer_clean = row[f'{correct_opt}_clean']
    w2v_d = w2v_distractors(answer_clean, row['article_clean'], w2v_model)
    print(f"W2V distractors for '{answer_clean[:40]}': {w2v_d}")

Word2Vec model not found at expected path. Skipping W2V distractors.
To enable: add 'GoogleNews vectors negative300' dataset to Kaggle input.


## 8. Extractive Hint Generation — Cosine Scoring

In [10]:
def sentence_tokenize(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', str(text)) if len(s.strip()) > 10]

def extractive_hints(article_clean: str, question_clean: str,
                      vec, top_k: int = 3) -> list:
    """
    Score each passage sentence by cosine similarity to the question;
    return top-K sentences as ranked hints.
    """
    sents = sentence_tokenize(article_clean)
    if not sents:
        return []
    q_vec   = vec.transform([question_clean])
    s_vecs  = vec.transform(sents)
    scores  = cosine_similarity(q_vec, s_vecs)[0]
    ranked  = np.argsort(scores)[::-1]
    return [sents[i] for i in ranked[:top_k]]

# Demo
row = val_df.iloc[3]
hints = extractive_hints(row['article_clean'], row['question_clean'], ohe_vec)
print("=== Extractive Hints Demo ===")
print(f"Question: {row['question_clean'][:100]}")
for i, h in enumerate(hints):
    print(f"  Hint {i+1}: {h[:120]}")

=== Extractive Hints Demo ===
Question: the purpose of education is_
  Hint 1: education is not an end but a means to an end in other words we do not educate children only for the purpose of educatin


## 9. ML-Scored Hint Generation — Logistic Regression on Sentence Features

In [11]:
def sentence_features(sent: str, question_clean: str,
                       article_clean: str, pos_in_doc: float,
                       vec) -> np.ndarray:
    """Feature vector for a sentence as a hint candidate."""
    q_words = set(question_clean.split()) - STOP_WORDS
    s_words = set(sent.split()) - STOP_WORDS

    kw_overlap = len(q_words & s_words) / (len(q_words) + 1)

    # Cosine similarity (OHE)
    q_vec = vec.transform([question_clean])
    s_vec = vec.transform([sent])
    cos   = float(cosine_similarity(q_vec, s_vec)[0, 0])

    sent_len_words = len(sent.split())
    sent_len_chars = len(sent)

    # Proximity to start of document (earlier sentences often more relevant)
    pos_bias = 1.0 - pos_in_doc  # 1=first, 0=last

    return np.array([kw_overlap, cos, sent_len_words,
                     sent_len_chars, pos_bias, pos_in_doc])

HINT_FEAT_COLS = ['kw_overlap', 'cos_sim', 'sent_len_words',
                  'sent_len_chars', 'pos_bias', 'pos_in_doc']

print("Building hint scorer dataset (3000 samples) ...")
HINT_SAMPLE = 3000
Xh, yh = [], []

for _, row in train_df.head(HINT_SAMPLE).iterrows():
    sents = sentence_tokenize(row['article_clean'])
    n = len(sents)
    if n == 0:
        continue
    correct_opt  = row['answer']
    answer_clean = row[f'{correct_opt}_clean']
    answer_words = set(answer_clean.split())

    for pos, sent in enumerate(sents):
        feat = sentence_features(sent, row['question_clean'],
                                  row['article_clean'],
                                  pos / n, ohe_vec)
        # Label: 1 if sentence contains content words from the correct answer
        s_words = set(sent.split()) - STOP_WORDS
        ans_content = answer_words - STOP_WORDS
        label = 1 if len(s_words & ans_content) >= max(1, len(ans_content) * 0.4) else 0
        Xh.append(feat)
        yh.append(label)

Xh = np.array(Xh)
yh = np.array(yh)
print(f"Hint dataset: {Xh.shape}, positive rate: {yh.mean():.3f}")

Building hint scorer dataset (3000 samples) ...
Hint dataset: (3000, 6), positive rate: 0.737


In [12]:
from sklearn.model_selection import train_test_split

Xh_tr, Xh_val, yh_tr, yh_val = train_test_split(
    Xh, yh, test_size=0.2, random_state=42, stratify=yh
)

# Classification scorer
print("Training hint scorer (Logistic Regression) ...")
hint_lr = LogisticRegression(C=1.0, class_weight='balanced',
                              max_iter=300, random_state=42)
hint_lr.fit(Xh_tr, yh_tr)
yh_pred = hint_lr.predict(Xh_val)

h_prec = precision_score(yh_val, yh_pred, zero_division=0)
h_rec  = recall_score(yh_val, yh_pred, zero_division=0)
h_f1   = f1_score(yh_val, yh_pred, zero_division=0)
h_acc  = accuracy_score(yh_val, yh_pred)
print(f"  Hint LR — Prec={h_prec:.4f}  Rec={h_rec:.4f}  "
      f"F1={h_f1:.4f}  Acc={h_acc:.4f}")

# Regression scorer for R² metric (predict relevance score 0-1)
print("Training hint REGRESSION scorer (Ridge) ...")
# Soft label = cosine similarity value
yh_soft = Xh[:, 1]  # use cosine similarity as soft relevance target
Xh_tr_r, Xh_val_r, yh_tr_r, yh_val_r = train_test_split(
    Xh, yh_soft, test_size=0.2, random_state=42
)
ridge = Ridge(alpha=1.0)
ridge.fit(Xh_tr_r, yh_tr_r)
yh_pred_r = ridge.predict(Xh_val_r)
r2   = r2_score(yh_val_r, yh_pred_r)
rmse = np.sqrt(mean_squared_error(yh_val_r, yh_pred_r))
print(f"  Ridge R²={r2:.4f}  RMSE={rmse:.4f}")

joblib.dump(hint_lr, f'{MOUT}/hint_lr_scorer.pkl')
joblib.dump(ridge,   f'{MOUT}/hint_ridge_scorer.pkl')
print("Hint scorers saved.")

Training hint scorer (Logistic Regression) ...
  Hint LR — Prec=0.7565  Rec=0.5271  F1=0.6213  Acc=0.5267
Training hint REGRESSION scorer (Ridge) ...
  Ridge R²=0.9826  RMSE=0.0082
Hint scorers saved.


## 10. ML-Scored Hint Generation Function

In [13]:
def ml_hints(article_clean: str, question_clean: str,
              scorer, vec, top_k: int = 3) -> list:
    """
    Score each sentence with the trained hint scorer;
    return top-K sentences as graduated hints
    (sorted from most general to most specific by score).
    """
    sents = sentence_tokenize(article_clean)
    if not sents:
        return []
    n = len(sents)
    feats = np.array([
        sentence_features(s, question_clean, article_clean, i / n, vec)
        for i, s in enumerate(sents)
    ])
    if hasattr(scorer, 'predict_proba'):
        scores = scorer.predict_proba(feats)[:, 1]
    else:
        scores = scorer.predict(feats)
    ranked = np.argsort(scores)[::-1]
    # Return from least specific (lower score) to most specific (highest)
    top_idx = ranked[:top_k][::-1]
    return [sents[i] for i in top_idx]

# Demo
row = val_df.iloc[3]
ml_h = ml_hints(row['article_clean'], row['question_clean'], hint_lr, ohe_vec)
print("=== ML-Scored Hints Demo ===")
print(f"Question: {row['question_clean'][:100]}")
for i, h in enumerate(ml_h, 1):
    print(f"  Hint {i}: {h[:120]}")

=== ML-Scored Hints Demo ===
Question: the purpose of education is_
  Hint 1: education is not an end but a means to an end in other words we do not educate children only for the purpose of educatin


## 11. Batch Evaluation on Validation Set

In [15]:
print("Running batch evaluation on val set (200 samples) ...")
EVAL_N = 200

dist_precision_scores, dist_recall_scores = [], []
hint_precision_scores = []
distractor_ranker_correct = 0

for _, row in val_df.head(EVAL_N).iterrows():
    correct_opt  = row['answer']
    
    # FIX 1: Ensure answer is a valid string
    raw_ans = row[f'{correct_opt}_clean']
    answer_clean = str(raw_ans) if pd.notna(raw_ans) else ""
    
    # FIX 2: Ensure article and question are strings
    article_clean = str(row['article_clean']) if pd.notna(row['article_clean']) else ""
    question_clean = str(row['question_clean']) if pd.notna(row['question_clean']) else ""

    gold_dist = set()
    for opt in ['A', 'B', 'C', 'D']:
        if opt != correct_opt:
            val = row[f'{opt}_clean']
            # FIX 3: Only add valid strings to gold distractors
            if isinstance(val, str):
                gold_dist.add(val)

    generated = generate_distractors(
        article_clean, answer_clean, rf_ranker, ohe_vec
    )

    # Precision: fraction of generated distractors that match a gold distractor
    matches = 0
    for g in generated:
        g_words = set(str(g).split()) # Safe split
        for gd in gold_dist:
            gd_words = set(gd.split())
            if len(g_words & gd_words) / (len(g_words | gd_words) + 1e-9) > 0.4:
                matches += 1
                break

    prec = matches / len(generated) if generated else 0
    rec  = matches / len(gold_dist) if gold_dist else 0
    dist_precision_scores.append(prec)
    dist_recall_scores.append(rec)

    # Distractor ranker accuracy: top-1 is NOT correct answer
    if generated and generated[0] != answer_clean:
        distractor_ranker_correct += 1

    # Hint precision: fraction of hints that overlap with gold answer sentence
    hints = ml_hints(article_clean, question_clean, hint_lr, ohe_vec, top_k=3)
    ans_words = set(answer_clean.split()) - STOP_WORDS
    
    h_match = 0
    for h in hints:
        # FIX 4: Ensure hint is a string
        if isinstance(h, str):
            if len(set(h.split()) & ans_words) >= max(1, int(len(ans_words) * 0.3)):
                h_match += 1
                
    hint_precision_scores.append(h_match / len(hints) if hints else 0)

avg_prec = np.mean(dist_precision_scores)
avg_rec  = np.mean(dist_recall_scores)
avg_f1   = 2 * avg_prec * avg_rec / (avg_prec + avg_rec + 1e-9)
rank_acc = distractor_ranker_correct / EVAL_N
hint_prec = np.mean(hint_precision_scores)

print(f"\n=== Model B Evaluation ({EVAL_N} val samples) ===")
print(f"Distractor Precision  : {avg_prec:.4f}")
print(f"Distractor Recall     : {avg_rec:.4f}")
print(f"Distractor F1         : {avg_f1:.4f}")
print(f"Ranker Accuracy       : {rank_acc:.4f}  (top-1 ≠ correct answer)")
print(f"Hint Precision        : {hint_prec:.4f}")

# Make sure 'r2' is defined from the earlier cell where the Ridge model was trained. 
# If it throws an error here, you must run the Ridge regression training cell first!
try:
    print(f"Ridge R²              : {r2:.4f}")
except NameError:
    r2 = 0.0 # Fallback in case cell 9 wasn't run
    print("Ridge R²              : Not calculated (Run Ridge regression cell first)")

eval_results = {
    'distractor_precision': avg_prec, 'distractor_recall': avg_rec,
    'distractor_f1': avg_f1, 'ranker_accuracy': rank_acc,
    'hint_precision': hint_prec, 'ridge_r2': r2
}
pd.DataFrame([eval_results]).to_csv(f'{MOUT}/model_b_results.csv', index=False)

Running batch evaluation on val set (200 samples) ...

=== Model B Evaluation (200 val samples) ===
Distractor Precision  : 0.0150
Distractor Recall     : 0.0150
Distractor F1         : 0.0150
Ranker Accuracy       : 1.0000  (top-1 ≠ correct answer)
Hint Precision        : 0.8500
Ridge R²              : 0.9826


## 12. Confusion Matrix & Visualizations

In [16]:
cm_ranker = confusion_matrix(yr_val, yr_pred_rf)
cm_hint   = confusion_matrix(yh_val, yh_pred)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(cm_ranker, annot=True, fmt='d', cmap='Oranges', ax=axes[0])
axes[0].set_title('Distractor Ranker (RF) — Val CM')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm_hint, annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('Hint Scorer (LR) — Val CM')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

# Metrics bar chart
metric_names = ['Dist-Prec', 'Dist-Rec', 'Dist-F1', 'Ranker-Acc', 'Hint-Prec']
metric_vals  = [avg_prec, avg_rec, avg_f1, rank_acc, hint_prec]
axes[2].barh(metric_names, metric_vals, color='steelblue')
axes[2].set_xlim(0, 1)
axes[2].set_title('Model B — Summary Metrics')
for i, v in enumerate(metric_vals):
    axes[2].text(v + 0.01, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.savefig(f'{MOUT}/model_b_performance.png', dpi=120)
plt.show()
print("Plot saved.")

Plot saved.


In [17]:
print("\n✅ Model B training complete. Saved files:")
for f in sorted(os.listdir(MOUT)):
    size = os.path.getsize(os.path.join(MOUT, f))
    print(f"  {f:40s}  {size/1024:.1f} KB")


✅ Model B training complete. Saved files:
  distractor_lr_ranker.pkl                  0.9 KB
  distractor_rf_ranker.pkl                  1072.5 KB
  hint_lr_scorer.pkl                        0.9 KB
  hint_ridge_scorer.pkl                     0.6 KB
  model_b_performance.png                   60.1 KB
  model_b_results.csv                       0.2 KB


In [20]:
import shutil
from IPython.display import FileLink

# 1. Define the target directory and the output zip name
folder_to_zip = '/kaggle/working/models'
output_filename = '/kaggle/working/all_models_backup' 

# 2. Create the zip archive
shutil.make_archive(output_filename, 'zip', folder_to_zip)
print(f"Successfully created {output_filename}.zip")

# 3. FIX: Generate the link using ONLY the relative filename
display(FileLink('all_models_backup.zip'))

Successfully created /kaggle/working/all_models_backup.zip


/kaggle/working/all_models_backup.zip